# Notebook 02 — Baseline Classifier Evaluation (`EfficientNet-B4`, Global View `A0`)

**Purpose:** Document the frozen `EfficientNet-B4` baseline protocol and preprocessing pipeline, inspect canonical held-out test metrics (`Top-1 Accuracy`, `Balanced Accuracy`, `Macro OvR AUROC`, `ECE`, `NLL`, `Brier`), and recompute discrimination, confusion matrix, and temperature scaling directly from frozen predictions when locally available.  
**Data & Artifact Requirement:** Aggregate baseline tables execute standalone. Live metric recomputation and the 7x7 confusion matrix require excluded local prediction arrays `research/baseline_v2/runs/clean_run_001/predictions/{test_canonical.npz,calibration_canonical.npz}` (`REQUIRES_MODEL_ARTIFACT`). This notebook never retrains the classifier.

## 1. Learning Objectives & Scientific Context

### Learning Objectives
1. Inspect the verified `EfficientNet-B4` (`17.9M` params) training and `A0` preprocessing pipeline (`get_original_transform()` vs `get_view_transform()`).
2. Verify global `A0` discrimination and calibration metrics (`69.21%` Top-1 Accuracy, `68.83%` Balanced Accuracy, `0.9274` Macro OvR AUROC, `2.52%` raw ECE vs `2.71%` temperature-scaled ECE).
3. When local frozen `.npz` logits exist, recompute all metrics, fit post-hoc BFGS temperature scaling ($T_{\mathrm{full}} = 0.851796$), and render the 7x7 class confusion matrix without modifying any frozen checkpoint.

### Prerequisites
- Core dependencies (`torch`, `scikit-learn`, `scipy`, `albumentations`, `matplotlib`).
- Optional local artifacts (`REQUIRES_MODEL_ARTIFACT`): `calibration_canonical.npz`, `test_canonical.npz`, and `epoch=09-val_auroc=0.0000.ckpt` (SHA-256 `0d14579d668d02781874cf91ad493fac4bf33ed53e3be928e3a7f9bb856ebbbb`).

## 2. Environment & Frozen Prediction Artifact Preflight

In [ ]:
from __future__ import annotations

import json
import os
import platform
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt


def find_repo_root(start: Path | None = None) -> Path:
    """Locate repository root portably without hardcoded machine paths."""
    cur = (start or Path.cwd()).resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "research" / "derma_act" / "config.py").is_file():
            return candidate
    raise FileNotFoundError("Could not locate DERMA-ACT repository root containing pyproject.toml and research/derma_act/config.py")


def find_tables_dir(repo_root: Path) -> Path | None:
    for rel in [
        Path("supplementary") / "tables",
        Path("manuscripts") / "current" / "PAPER_DRAFT_Q2_V3_2_1_PRE_SUBMISSION" / "tables",
    ]:
        cand = repo_root / rel
        if cand.is_dir():
            return cand
    return None


SEED = 42
random.seed(SEED)
np.random.seed(SEED)

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import hashlib
from research.derma_act.config import CLASS_NAMES, CLASSIFIER_PATH, EXPECTED_HASHES
from research.derma_act.core.transforms import get_original_transform, get_view_transform

OUT_DIR = REPO_ROOT / "outputs" / "notebooks" / "nb02_baseline"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_NPZ = REPO_ROOT / "research" / "baseline_v2" / "runs" / "clean_run_001" / "predictions" / "test_canonical.npz"
CAL_NPZ = REPO_ROOT / "research" / "baseline_v2" / "runs" / "clean_run_001" / "predictions" / "calibration_canonical.npz"
ACT_METRICS_CSV = REPO_ROOT / "research" / "derma_act" / "p5_6_action_quality_audit" / "P5_6_ACTION_PREDICTION_METRICS.csv"
TABLES_DIR = find_tables_dir(REPO_ROOT)

HAS_FROZEN_PREDS = TEST_NPZ.is_file() and CAL_NPZ.is_file()
ckpt_exists = CLASSIFIER_PATH.is_file()
ckpt_sha_ok = None
if ckpt_exists:
    actual_sha = hashlib.sha256(CLASSIFIER_PATH.read_bytes()).hexdigest()
    ckpt_sha_ok = actual_sha == EXPECTED_HASHES["classifier"]

print("A0 original transform pipeline:", [type(t).__name__ for t in get_original_transform().transforms])
print("A1-A9 view transform pipeline :", [type(t).__name__ for t in get_view_transform().transforms])
display(pd.DataFrame([
    {"artifact": "test_canonical.npz (N=3,686)", "available": TEST_NPZ.is_file(), "tier": "REQUIRES_MODEL_ARTIFACT"},
    {"artifact": "calibration_canonical.npz (N=1,401)", "available": CAL_NPZ.is_file(), "tier": "REQUIRES_MODEL_ARTIFACT"},
    {"artifact": "EfficientNet-B4 checkpoint (.ckpt)", "available": ckpt_exists, "tier": f"SHA256_MATCH={ckpt_sha_ok}"},
    {"artifact": "P5_6_ACTION_PREDICTION_METRICS.csv", "available": ACT_METRICS_CSV.is_file(), "tier": "TRACKED"},
]))

## 3. Baseline Protocol & Tracked Canonical Summary

The frozen backbone is `timm/efficientnet_b4.ra2_in1k` (`17.9M` parameters) with classification head `Flatten -> BatchNorm1d(1792) -> Dropout(0.3) -> Linear(1792, 7)`, trained on ISIC 2019 Train (`N = 16,702`) with `WeightedClassSampler`, `CrossEntropyLoss(label_smoothing=0.1)`, `AdamW(lr=1e-4, weight_decay=1e-4)`, and `CosineAnnealingLR(T_max=50)`.

Evaluating `A0` applies `Resize(291) -> CenterCrop(224) -> ImageNet Normalize` (`get_original_transform()`). Crucially, virtual views `A1`–`A9` are already `224x224` and must use `get_view_transform()` (no second center crop).

In [ ]:
import re

act_metrics = pd.read_csv(ACT_METRICS_CSV)
a0_row = act_metrics[act_metrics["action_token"] == "A0"].iloc[0]
print(f"Tracked A0 standalone accuracy (per-action tensor): {a0_row['standalone_accuracy']:.6f} | Mean entropy: {a0_row['mean_entropy_nats']:.4f} nats | Mean top-1 conf: {a0_row['mean_top1_confidence']:.4f}")

# Parse authoritative per-class Panel B from tab2_baseline_performance.tex
tab2_rows = []
if TABLES_DIR is not None and (TABLES_DIR / "tab2_baseline_performance.tex").is_file():
    for ln in (TABLES_DIR / "tab2_baseline_performance.tex").read_text(encoding="utf-8").splitlines():
        if re.match(r"^(MEL|NV|BCC|AKIEC|BKL|DF|VASC)", ln.strip()):
            parts = [p.strip() for p in ln.replace("\\\\", "").split("&")]
            cls_name = re.sub(r"\$.*\$", "", parts[0]).strip()
            sens_val = float(re.search(r"([0-9.]+)\\%", parts[3]).group(1))
            tab2_rows.append({
                "class": cls_name,
                "support_n": int(parts[1].replace(",", "")),
                "true_positives": int(parts[2].replace(",", "")),
                "sensitivity_pct": sens_val,
                "specificity_pct": float(parts[4].replace("\\%", "")),
                "precision_pct": float(parts[5].replace("\\%", "")),
                "ovr_auroc": float(parts[6]),
            })
tab2_df = pd.DataFrame(tab2_rows)
display(tab2_df)

## 4. Live Recomputation from Frozen Predictions & Temperature Scaling (`REQUIRES_MODEL_ARTIFACT`)

When `calibration_canonical.npz` and `test_canonical.npz` are present locally, the cell below recomputes Top-1 accuracy, Balanced accuracy, Macro OvR AUROC, 15-bin ECE, NLL, Multi-class Brier score, BFGS temperature scaling $T_{\mathrm{full}}$, and the 7x7 confusion matrix directly from frozen arrays.

In [ ]:
from scipy.optimize import minimize
from scipy.special import softmax
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, roc_auc_score


def compute_ece_15(probs: np.ndarray, labels: np.ndarray, n_bins: int = 15) -> float:
    conf = probs.max(axis=1)
    preds = probs.argmax(axis=1)
    acc = (preds == labels).astype(float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (conf > edges[i]) & (conf <= edges[i + 1]) if i > 0 else (conf >= edges[i]) & (conf <= edges[i + 1])
        if mask.any():
            ece += (mask.sum() / len(labels)) * abs(acc[mask].mean() - conf[mask].mean())
    return float(ece)


def compute_nll_brier(probs: np.ndarray, labels: np.ndarray) -> tuple[float, float]:
    n = len(labels)
    p_true = np.clip(probs[np.arange(n), labels], 1e-12, 1.0)
    nll = float(-np.log(p_true).mean())
    onehot = np.eye(probs.shape[1])[labels]
    brier = float(np.sum((probs - onehot) ** 2, axis=1).mean())
    return nll, brier


cm_norm = None
live_metrics_df = None

if HAS_FROZEN_PREDS:
    cal = np.load(CAL_NPZ, allow_pickle=True)
    tst = np.load(TEST_NPZ, allow_pickle=True)
    y_cal, z_cal = cal["true_label"].astype(int), cal["logits"].astype(np.float64)
    y_tst, z_tst, p_raw = tst["true_label"].astype(int), tst["logits"].astype(np.float64), tst["probabilities"].astype(np.float64)

    # Fit temperature T_full via BFGS NLL minimization on D_cal (N=1,401)
    def cal_nll(log_t):
        t = float(np.exp(log_t[0]))
        p = softmax(z_cal / t, axis=1)
        return float(-np.log(np.clip(p[np.arange(len(y_cal)), y_cal], 1e-12, 1.0)).mean())

    t_full = float(np.exp(minimize(cal_nll, [0.0], method="BFGS").x[0]))
    p_ts = softmax(z_tst / t_full, axis=1)

    rows = []
    for label_name, probs in [("Raw Softmax (A0)", p_raw), (f"Temp-Scaled (T={t_full:.6f})", p_ts)]:
        preds = probs.argmax(axis=1)
        nll, brier = compute_nll_brier(probs, y_tst)
        rows.append({
            "variant": label_name,
            "top1_acc_pct": round(100.0 * (preds == y_tst).mean(), 2),
            "balanced_acc_pct": round(100.0 * balanced_accuracy_score(y_tst, preds), 2),
            "macro_ovr_auroc": round(float(roc_auc_score(y_tst, probs, multi_class="ovr", average="macro")), 4),
            "ece_15bin_pct": round(100.0 * compute_ece_15(probs, y_tst), 2),
            "nll": round(nll, 4),
            "brier": round(brier, 4),
        })
    live_metrics_df = pd.DataFrame(rows)
    display(live_metrics_df)

    cm = confusion_matrix(y_tst, p_raw.argmax(axis=1), labels=list(range(7)))
    cm_norm = cm / cm.sum(axis=1, keepdims=True)
else:
    print("SKIPPED (REQUIRES_MODEL_ARTIFACT): Frozen prediction NPZs are excluded from public Git.")
    print("Required local paths (derived from CC-BY-NC 4.0 ISIC 2019):")
    print("  research/baseline_v2/runs/clean_run_001/predictions/calibration_canonical.npz")
    print("  research/baseline_v2/runs/clean_run_001/predictions/test_canonical.npz")
    print(f"Expected classifier SHA-256: {EXPECTED_HASHES['classifier']}")

## 5. Visualization: Per-Class Operating Characteristics & Confusion Matrix

In [ ]:
if cm_norm is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
    im = axes[0].imshow(cm_norm, cmap="Blues", vmin=0.0, vmax=1.0)
    labels_upper = [c.upper() for c in CLASS_NAMES]
    axes[0].set_xticks(range(7)); axes[0].set_xticklabels(labels_upper, rotation=45)
    axes[0].set_yticks(range(7)); axes[0].set_yticklabels(labels_upper)
    axes[0].set_xlabel("Predicted Class"); axes[0].set_ylabel("True Class")
    axes[0].set_title("Held-Out Test A0 Row-Normalized Confusion Matrix (N=3,686)")
    for i in range(7):
        for j in range(7):
            val = cm_norm[i, j]
            axes[0].text(j, i, f"{val:.2f}", ha="center", va="center", color="white" if val > 0.5 else "black", fontsize=8)
    fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)
    ax_bar = axes[1]
else:
    fig, ax_bar = plt.subplots(figsize=(7.5, 4.4))

if not tab2_df.empty:
    x = np.arange(len(tab2_df))
    ax_bar.bar(x - 0.18, tab2_df["sensitivity_pct"], width=0.36, label="Sensitivity / Recall (%)", color="#2b6cb0")
    ax_bar.bar(x + 0.18, tab2_df["ovr_auroc"] * 100.0, width=0.36, label="OvR AUROC (x100)", color="#38a169")
    ax_bar.set_xticks(x)
    ax_bar.set_xticklabels(tab2_df["class"])
    ax_bar.set_ylim(0, 108)
    ax_bar.set_ylabel("Percentage / Score x100")
    ax_bar.set_title("Per-Class Baseline A0 Recall vs. OvR AUROC (N=3,686)")
    ax_bar.legend(loc="lower right")
    ax_bar.grid(axis="y", alpha=0.3)

plt.tight_layout()
fig.savefig(OUT_DIR / "baseline_a0_evaluation.png", dpi=150)
plt.show()

## 6. Interpretation & Limitations

- **Class Sensitivity Disparity:** While Macro OvR AUROC reaches `0.9274`, single-view Top-1 sensitivity is only `45.25%` on `AKIEC` (`n = 221`) and `58.31%` on `MEL` (`n = 686`). Sparse strata `DF` (`n = 25`) and `VASC` (`n = 19`) have wide cluster-stratified confidence intervals and low precision (`18.48%` and `30.51%`).
- **Temperature Scaling Trade-Off:** Because $T_{\mathrm{full}} = 0.851796 < 1$ sharpens logits slightly, test NLL decreases (`0.8273 -> 0.8174`) while 15-bin ECE increases slightly (`2.52% -> 2.71%`). Top-1 accuracy and balanced accuracy are strictly invariant under positive scalar temperature scaling.
- **No Retraining / Non-Recoverable Pre-Freeze Seeds:** Historical cloud training stop triggers and exact pre-freeze PyTorch optimization seeds are not recoverable locally; reproducibility is anchored on frozen checkpoints and canonical `.npz` logits.

### Reproduction & References
- Output saved to: `outputs/notebooks/nb02_baseline/baseline_a0_evaluation.png`
- Related docs: [Model Provenance](../provenance/MODEL_PROVENANCE.md) · [Artifacts Registry](../artifacts/README.md) · [Architecture](../docs/ARCHITECTURE.md)